# Lead–lag analysis of news coverage and OGD portal search — pipeline

Imports the verified `var_granger.py` module and reproduces the full analysis (main + sensitivity VAR/Granger, IRF/FEVD, stability, Table A2, Section 2.2 correlation, Figure A1).

> Korean is retained only for **raw-data file columns** (`연도`, `월`, `검색어`, `검색 건수`, `특성추출(...)`, `일자`) and **keyword values** (analysis units, e.g. `반려동물`). `df_adf_results` columns are English.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings

from statsmodels.tools.sm_exceptions import ValueWarning
for w in (FutureWarning, UserWarning, ValueWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=w)

import var_granger as vg   # verified analysis module


In [ ]:
# --- paths (edit DATA_DIR to point to the source files) ---
DATA_DIR   = "data"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_MONTHS_MAIN = 21
MIN_MONTHS_SENS = 13
ADF_SIGLEVEL    = 0.05


## 2. Load data

Place the source files in `DATA_DIR`. Raw data is not redistributed (see README).

In [ ]:
# search workbook + news file (raw-data column names 월/연도/검색어/검색 건수/특성추출(...) kept to match the files)
raw_search = pd.read_excel(f"{DATA_DIR}/keyword_top50_2201_2505.xlsx")
raw_news   = pd.read_csv(f"{DATA_DIR}/news_merged.csv")
df_search  = raw_search.copy()
df_news    = raw_news.copy()


## 3. Preprocessing

News-coverage intensity, month pivot, and 1/√2 imputation.

In [ ]:
# skip if already recomputed
df_news_monthly = vg.calculate_news_intensity(df_news)

# unify index
df_news_monthly.index = pd.to_datetime(df_news_monthly.index).strftime('%Y-%m')

print(f"news intensity: {df_news_monthly.shape[0]} months x {df_news_monthly.shape[1]} keywords")
print(f"non-zero cells: {(df_news_monthly > 0).sum().sum():,}")

In [ ]:
import re

def safe_extract_month(x):
    """Safely extract a 2-digit month from various month formats."""
    s = str(x).strip()
    if '-' in s:
        return s.split('-')[1][:2].zfill(2)
    m = re.findall(r'\d+', s)
    if m:
        num = m[0]
        if len(num) >= 4:
            return num[-2:]
        else:
            return num.zfill(2)
    return '01'

# build year-month
df_search['safe_month'] = df_search['월'].apply(safe_extract_month)
df_search['year_month'] = df_search['연도'].astype(str) + '-' + df_search['safe_month']

# raw pivot (keep zeros; observed-month counts, Table 3)
df_search_raw_pivot = df_search.pivot_table(
    index='year_month', columns='검색어', values='검색 건수',
    aggfunc='sum'
).fillna(0)
df_search_raw_pivot.index = pd.to_datetime(df_search_raw_pivot.index)\
                              .strftime('%Y-%m')

print(f"raw search pivot: {df_search_raw_pivot.shape[0]} months x {df_search_raw_pivot.shape[1]} keywords")

In [ ]:
df_search_imputed = vg.apply_lod_imputation(df_search_raw_pivot)

# checks
assert (df_search_imputed > 0).all().all(), "zero or negative value after LOD imputation"
print(f"imputed search pivot: {df_search_imputed.shape}")
print(f"min (should be > 0): {df_search_imputed.values.min():.4f}")

Align news and search on common months and shared keywords.

In [ ]:
# common time index
common_index = df_news_monthly.index.intersection(df_search_imputed.index)

# shared keywords (sorted)
matched_keywords = sorted(
    set(df_news_monthly.columns) & set(df_search_imputed.columns)
)

# build matched DataFrames
df_news_matched   = df_news_monthly.loc[common_index, matched_keywords]
df_search_matched = df_search_imputed.loc[common_index, matched_keywords]

# checks
assert df_news_matched.shape == df_search_matched.shape, "matched shape mismatch"
assert list(df_news_matched.index) == list(df_search_matched.index), "index mismatch"
assert list(df_news_matched.columns) == list(df_search_matched.columns), "column mismatch"

print(f"common period: {len(common_index)} months")
print(f"matched keywords: {len(matched_keywords)}")
print(f"df_news_matched:   {df_news_matched.shape}")
print(f"df_search_matched: {df_search_matched.shape}")

## 4. Stationarity (ADF) and keyword selection

`df_adf_results` columns: `keyword`, `search_status`, `search_diff`, `news_status`, `news_diff`, `passed`.

In [ ]:
adf_records = []
passed_keywords = []
failed_keywords = []

for kw in matched_keywords:
    # log transform
    s_log = np.log(df_search_matched[kw])        # search volume
    n_log = np.log1p(df_news_matched[kw])         # news (ln(x+1))
    
    s_ser, s_diffed, s_status = vg.adf_test(s_log, sig=ADF_SIGLEVEL)
    n_ser, n_diffed, n_status = vg.adf_test(n_log, sig=ADF_SIGLEVEL)
    
    adf_records.append({
        'keyword': kw,
        'search_status': s_status,
        'search_diff': s_diffed,
        'news_status': n_status,
        'news_diff': n_diffed,
        'passed': (s_ser is not None) and (n_ser is not None)
    })
    
    if (s_ser is not None) and (n_ser is not None):
        passed_keywords.append(kw)
    else:
        failed_keywords.append(kw)

df_adf_results = pd.DataFrame(adf_records)

# save
df_adf_results.to_csv(f'{OUTPUT_DIR}/adf_results.csv',
                      index=False, encoding='utf-8-sig')

# summary
print(f"===== ADF test results =====")
print(f"total matched: {len(matched_keywords)}")
print(f"passed:  {len(passed_keywords)}")
print(f"failed:  {len(failed_keywords)}")
print(f"\n--- failure reasons ---")
failed_df = df_adf_results[~df_adf_results['passed']]
for _, row in failed_df.iterrows():
    reasons = []
    if row['search_status'] in ['nonstationary', 'zero_var', 'zero_var_diff']:
        reasons.append(f"search({row['search_status']})")
    if row['news_status'] in ['nonstationary', 'zero_var', 'zero_var_diff']:
        reasons.append(f"news({row['news_status']})")
    print(f"  {row['keyword']}: {', '.join(reasons)}")

Observed-month counts → main (≥21) and sensitivity (≥13) keyword sets.

In [ ]:
# observed months for ADF-passed keywords (from raw pivot)
appearance = (df_search_raw_pivot[passed_keywords] > 0).sum(axis=0)

# save
appearance.to_csv(f'{OUTPUT_DIR}/appearance_months.csv',
                  encoding='utf-8-sig', header=['observed_months'])

print(f"===== observed-month distribution (ADF-passed) =====")
print(f"keywords: {len(appearance)}")
print(f"mean: {appearance.mean():.1f} / median: {appearance.median():.0f} months")
print(f"min: {appearance.min()} / max: {appearance.max()}")

# buckets: 13-20 and 21-24 split out
bins   = [0, 3, 6, 12, 20, 24, 41]
labels = ['1-3 months', '4-6 months', '7-12 months',
          '13-20 months', '21-24 months', '25-41 months']

dist = pd.cut(appearance, bins=bins, labels=labels, right=True)\
         .value_counts().sort_index()

print(f"\n===== Table 3: observed-month distribution (ADF-passed) =====")
cum = 0
for label, count in dist.items():
    cum += count
    pct = count / len(appearance) * 100
    cum_pct = cum / len(appearance) * 100
    print(f"  {label:>12}: {count:>4} ({pct:>5.1f}%) / cumulative {cum_pct:>5.1f}%")

# check main / sensitivity keyword counts
n_main = (appearance >= MIN_MONTHS_MAIN).sum()
n_sens = (appearance >= MIN_MONTHS_SENS).sum()
n_main_check = dist['21-24 months'] + dist['25-41 months']
n_sens_check = dist['13-20 months'] + dist['21-24 months'] + dist['25-41 months']

print(f"\nmain (>= {MIN_MONTHS_MAIN} months): {n_main} (check: {n_main_check})")
print(f"sensitivity (>= {MIN_MONTHS_SENS} months): {n_sens} (check: {n_sens_check})")
assert n_main == n_main_check, "main keyword count mismatch"
assert n_sens == n_sens_check, "sensitivity keyword count mismatch"

analysis_keywords_main = sorted(appearance[appearance >= MIN_MONTHS_MAIN].index.tolist())
analysis_keywords_sens = sorted(appearance[appearance >= MIN_MONTHS_SENS].index.tolist())

# save
pd.Series(analysis_keywords_main, name='keyword')\
  .to_csv(f'{OUTPUT_DIR}/keywords_main.csv', index=False, encoding='utf-8-sig')
pd.Series(analysis_keywords_sens, name='keyword')\
  .to_csv(f'{OUTPUT_DIR}/keywords_sens.csv', index=False, encoding='utf-8-sig')

print(f"===== main keywords ({len(analysis_keywords_main)}) =====")
print(analysis_keywords_main)

print(f"\n===== sensitivity keywords ({len(analysis_keywords_sens)}) =====")
print(analysis_keywords_sens)

## 5. VAR estimation and Granger causality

In [ ]:
df_all_main = vg.run_var_granger_batch(
    analysis_keywords_main,
    df_search_matched,
    df_news_matched,
    df_adf_results,
    label='main'
)

# error filter
df_valid_main = df_all_main[df_all_main['error'].isna()].copy()
df_error_main = df_all_main[df_all_main['error'].notna()].copy()

print(f"\n===== main analysis summary (N={len(analysis_keywords_main)}) =====")
print(f"completed: {len(df_valid_main)}")
print(f"errors: {len(df_error_main)}")

if len(df_error_main) > 0:
    print(f"\n--- error keywords ---")
    for _, row in df_error_main.iterrows():
        print(f"  {row['keyword']}: {row['error']}")

In [ ]:
# optimal lag distribution
print(f"\n--- optimal lag distribution (main, N={len(df_valid_main)}) ---")
lag_dist_main = df_valid_main['optimal_lag'].value_counts().sort_index()
for lag, count in lag_dist_main.items():
    pct = count / len(df_valid_main) * 100
    print(f"  lag {lag}: {count} ({pct:.1f}%)")

# type distribution
print(f"\n--- Granger type distribution (main) ---")
type_order = ['N->S', 'S->N', 'bidirectional', 'independent']
for t in type_order:
    count = (df_valid_main['type'] == t).sum()
    pct = count / len(df_valid_main) * 100
    print(f"  {t}: {count} ({pct:.1f}%)")

# significant keyword details
print(f"\n--- significant keyword details (main) ---")
sig_main = df_valid_main[
    (df_valid_main['N2S_sig'] == True) | (df_valid_main['S2N_sig'] == True)
].copy()
print(f"significant keywords: {len(sig_main)}\n")

if len(sig_main) > 0:
    cols_show = ['keyword', 'optimal_lag', 'type',
                 'N2S_F', 'N2S_p', 'S2N_F', 'S2N_p']
    print(sig_main[cols_show].to_string(index=False))

In [ ]:
df_all_sens = vg.run_var_granger_batch(
    analysis_keywords_sens,
    df_search_matched,
    df_news_matched,
    df_adf_results,
    label='sensitivity'
)

# error filter
df_valid_sens = df_all_sens[df_all_sens['error'].isna()].copy()
df_error_sens = df_all_sens[df_all_sens['error'].notna()].copy()

print(f"\n===== sensitivity analysis summary (N={len(analysis_keywords_sens)}) =====")
print(f"completed: {len(df_valid_sens)}")
print(f"errors: {len(df_error_sens)}")

if len(df_error_sens) > 0:
    print(f"\n--- error keywords ---")
    for _, row in df_error_sens.iterrows():
        print(f"  {row['keyword']}: {row['error']}")

In [ ]:
# type distribution (main vs sensitivity)
print(f"\n--- type distribution: main vs sensitivity ---")
print(f"{'type':<12}{'main':>20}{'sensitivity':>20}")
print('-' * 52)
for t in type_order:
    c_main = (df_valid_main['type'] == t).sum()
    p_main = c_main / len(df_valid_main) * 100
    c_sens = (df_valid_sens['type'] == t).sum()
    p_sens = c_sens / len(df_valid_sens) * 100
    print(f"{t:<12}{c_main:>5} ({p_main:>4.1f}%)      {c_sens:>5} ({p_sens:>4.1f}%)")

# sensitivity significant keywords
print(f"\n--- sensitivity significant keyword details ---")
sig_sens = df_valid_sens[
    (df_valid_sens['N2S_sig'] == True) | (df_valid_sens['S2N_sig'] == True)
].copy()
print(f"significant keywords: {len(sig_sens)}\n")

if len(sig_sens) > 0:
    cols_show = ['keyword', 'optimal_lag', 'type',
                 'N2S_F', 'N2S_p', 'S2N_F', 'S2N_p']
    print(sig_sens[cols_show].to_string(index=False))

## 6. Impulse response and forecast-error variance decomposition

N→S FEVD at h=10 (target=Search, note=News): pet 56.2%, real estate 43.2%, tourism 12.4%.

In [ ]:
df_irf_main, df_fevd_main = vg.run_irf_fevd_batch(
    df_valid_main, df_search_matched, df_news_matched, df_adf_results,
    label='main', horizon=10
)

In [ ]:
# FEVD at h=10, N_first (default)
fevd10_nf = df_fevd_main[
    (df_fevd_main['horizon'] == 10) & (df_fevd_main['ordering'] == 'N_first')
].copy()

print(f"\n===== FEVD summary (h=10, N_first) =====")
print(f"{'keyword':<10} {'N→S(%)':>10} {'S→N(%)':>10}")
print('-' * 32)

# iterate over the 3 main significant keywords
sig_main = df_valid_main[
    (df_valid_main['N2S_sig'] == True) | (df_valid_main['S2N_sig'] == True)
]['keyword'].tolist()

for kw in sig_main:
    rows = fevd10_nf[fevd10_nf['keyword'] == kw]
    n2s = rows[(rows['target'] == 'Search') & (rows['note'] == 'News')]['FEVD'].values
    s2n = rows[(rows['target'] == 'News') & (rows['note'] == 'Search')]['FEVD'].values
    n2s_val = n2s[0] * 100 if len(n2s) > 0 else None
    s2n_val = s2n[0] * 100 if len(s2n) > 0 else None
    print(f"{kw:<10} {n2s_val:>9.1f}% {s2n_val:>9.1f}%")

In [ ]:
# N_first vs S_first comparison (h=10)
fevd10_sf = df_fevd_main[
    (df_fevd_main['horizon'] == 10) & (df_fevd_main['ordering'] == 'S_first')
].copy()

print(f"\n===== Cholesky ordering robustness (h=10, N_first vs S_first) =====")
print(f"{'keyword':<10} {'N→S (Nf)':>10} {'N→S (Sf)':>10}  "
      f"{'S→N (Nf)':>10} {'S→N (Sf)':>10}")
print('-' * 60)

for kw in sig_main:
    nf = fevd10_nf[fevd10_nf['keyword'] == kw]
    sf = fevd10_sf[fevd10_sf['keyword'] == kw]
    
    n2s_nf = nf[(nf['target'] == 'Search') & (nf['note'] == 'News')]['FEVD'].values
    n2s_sf = sf[(sf['target'] == 'Search') & (sf['note'] == 'News')]['FEVD'].values
    s2n_nf = nf[(nf['target'] == 'News') & (nf['note'] == 'Search')]['FEVD'].values
    s2n_sf = sf[(sf['target'] == 'News') & (sf['note'] == 'Search')]['FEVD'].values
    
    def pct(arr):
        return arr[0] * 100 if len(arr) > 0 else float('nan')
    
    print(f"{kw:<10} {pct(n2s_nf):>9.1f}% {pct(n2s_sf):>9.1f}%  "
          f"{pct(s2n_nf):>9.1f}% {pct(s2n_sf):>9.1f}%")

In [ ]:
# sensitivity-significant keywords not in the main set
sig_sens_all = df_valid_sens[
    (df_valid_sens['N2S_sig'] == True) | (df_valid_sens['S2N_sig'] == True)
]['keyword'].tolist()
sens_only = [kw for kw in sig_sens_all if kw not in sig_main]
print(f"sensitivity-only significant keywords: {sens_only}")

# compute over all sensitivity keywords (3 overlap with main)
df_irf_sens, df_fevd_sens = vg.run_irf_fevd_batch(
    df_valid_sens, df_search_matched, df_news_matched, df_adf_results,
    label='sensitivity', horizon=10
)

In [ ]:
# FEVD for sensitivity-only significant keywords (h=10)
fevd10_sens_nf = df_fevd_sens[
    (df_fevd_sens['horizon'] == 10) & (df_fevd_sens['ordering'] == 'N_first')
].copy()
fevd10_sens_sf = df_fevd_sens[
    (df_fevd_sens['horizon'] == 10) & (df_fevd_sens['ordering'] == 'S_first')
].copy()

print(f"\n===== FEVD for additional sensitivity-significant keywords (h=10) =====")
print(f"{'keyword':<8} {'type':<10} "
      f"{'N→S(Nf)':>10} {'N→S(Sf)':>10} "
      f"{'S→N(Nf)':>10} {'S→N(Sf)':>10}")
print('-' * 60)

for kw in sens_only:
    row = df_valid_sens[df_valid_sens['keyword'] == kw].iloc[0]
    kw_type = row['type']
    
    nf = fevd10_sens_nf[fevd10_sens_nf['keyword'] == kw]
    sf = fevd10_sens_sf[fevd10_sens_sf['keyword'] == kw]
    
    n2s_nf = nf[(nf['target'] == 'Search') & (nf['note'] == 'News')]['FEVD'].values
    n2s_sf = sf[(sf['target'] == 'Search') & (sf['note'] == 'News')]['FEVD'].values
    s2n_nf = nf[(nf['target'] == 'News') & (nf['note'] == 'Search')]['FEVD'].values
    s2n_sf = sf[(sf['target'] == 'News') & (sf['note'] == 'Search')]['FEVD'].values
    
    def pct(arr):
        return arr[0] * 100 if len(arr) > 0 else float('nan')
    
    print(f"{kw:<8} {kw_type:<10} "
          f"{pct(n2s_nf):>9.1f}% {pct(n2s_sf):>9.1f}% "
          f"{pct(s2n_nf):>9.1f}% {pct(s2n_sf):>9.1f}%")

In [ ]:
# track apartment FEVD across horizons (h=1..10)
if '아파트' in sens_only:
    print(f"\n===== apartment FEVD trajectory (N_first) =====")
    apt_nf = df_fevd_sens[
        (df_fevd_sens['keyword'] == '아파트') &
        (df_fevd_sens['ordering'] == 'N_first')
    ].sort_values(['horizon', 'target'])
    
    print(f"{'horizon':>4} {'N→S(%)':>10} {'S→N(%)':>10}")
    for h in range(1, 11):
        row_h = apt_nf[apt_nf['horizon'] == h]
        n2s = row_h[(row_h['target'] == 'Search') & (row_h['note'] == 'News')]['FEVD'].values
        s2n = row_h[(row_h['target'] == 'News') & (row_h['note'] == 'Search')]['FEVD'].values
        n2s_v = n2s[0] * 100 if len(n2s) > 0 else float('nan')
        s2n_v = s2n[0] * 100 if len(s2n) > 0 else float('nan')
        print(f"{h:>4} {n2s_v:>9.1f}% {s2n_v:>9.1f}%")
    
    print(f"\n===== apartment FEVD trajectory (S_first) =====")
    apt_sf = df_fevd_sens[
        (df_fevd_sens['keyword'] == '아파트') &
        (df_fevd_sens['ordering'] == 'S_first')
    ].sort_values(['horizon', 'target'])
    
    print(f"{'horizon':>4} {'N→S(%)':>10} {'S→N(%)':>10}")
    for h in range(1, 11):
        row_h = apt_sf[apt_sf['horizon'] == h]
        n2s = row_h[(row_h['target'] == 'Search') & (row_h['note'] == 'News')]['FEVD'].values
        s2n = row_h[(row_h['target'] == 'News') & (row_h['note'] == 'Search')]['FEVD'].values
        n2s_v = n2s[0] * 100 if len(n2s) > 0 else float('nan')
        s2n_v = s2n[0] * 100 if len(s2n) > 0 else float('nan')
        print(f"{h:>4} {n2s_v:>9.1f}% {s2n_v:>9.1f}%")

## 7. VAR stability check (companion-matrix eigenvalues)

In [ ]:
"""
VAR stability check: for all 26 main keywords,
verify companion-matrix eigenvalues lie inside the unit circle.

Following Lutkepohl (2005), Proposition 2.1.

Output:
    - all pass: justifies the manuscript statement
    - any fail: report keyword and eigenvalues
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.var_model import VAR


def check_var_stability(df_var, lag, tolerance=1.0, verbose=False):
    """
    Check the VAR stability condition.
    
    Parameters
    ----------
    df_var : pd.DataFrame
        VAR data.
    lag : int
        VAR lag order.
    tolerance : float, default 1.0
        stability threshold; stable if all |eigenvalue| < tolerance.
    verbose : bool
        whether to print eigenvalue details.
    
    Returns
    -------
    tuple (bool, np.ndarray, float)
        is_stable: stability flag
        eigenvalues: companion-matrix eigenvalues (complex)
        max_abs: maximum absolute eigenvalue
    """
    model = VAR(df_var)
    res = model.fit(lag)
    
    # companion-matrix eigenvalues (statsmodels)
    # roots are reciprocals of characteristic roots; |roots| > 1 => stable
    roots = res.roots
    
    # eigenvalue = 1/roots (per statsmodels docs)
    # is_stable method also available
    is_stable_sm = res.is_stable(verbose=verbose)
    
    # |eigenvalue| = 1/|roots|
    eigenvalues = 1.0 / roots
    max_abs = np.abs(eigenvalues).max()
    
    return is_stable_sm, eigenvalues, max_abs


def verify_all_keywords_stability(keywords_df, df_search, df_news, 
                                    adf_results):
    """
    Check VAR stability for all 26 main keywords.
    
    Parameters
    ----------
    keywords_df : pd.DataFrame
        loaded var_granger_results_main.csv.
        requires 'keyword' and 'optimal_lag' columns.
    df_search, df_news : pd.DataFrame
        wide-format data.
    adf_results : pd.DataFrame
        ADF results.
    
    Returns
    -------
    pd.DataFrame
        per-keyword stability results.
    """
    results = []
    
    for _, row in keywords_df.iterrows():
        kw = row['keyword']
        lag = int(row['optimal_lag'])
        
        try:
            df_var, _ = vg.prepare_keyword_data(
                kw, df_search, df_news, adf_results
            )
            
            is_stable, eigvals, max_abs = check_var_stability(
                df_var, lag, verbose=False
            )
            
            results.append({
                'keyword': kw,
                'lag': lag,
                'stability': 'pass' if is_stable else 'fail',
                'max_abs_eig': round(max_abs, 4),
                'n_eig': len(eigvals),
            })
            
        except Exception as e:
            results.append({
                'keyword': kw,
                'lag': lag,
                'stability': f'error: {str(e)[:40]}',
                'max_abs_eig': np.nan,
                'n_eig': np.nan,
            })
    
    return pd.DataFrame(results)


# ===== run =====
if __name__ == '__main__':
    # load main results
    keywords_df = pd.read_csv(
        f'{OUTPUT_DIR}/var_granger_results_main.csv'
    )
    
    print(f"keywords to check: {len(keywords_df)}\n")
    
    # run full check
    result_df = verify_all_keywords_stability(
        keywords_df=keywords_df,
        df_search=df_search_matched,
        df_news=df_news_matched,
        adf_results=df_adf_results,
    )
    
    # summary
    n_pass = (result_df['stability'] == 'pass').sum()
    n_total = len(result_df)
    
    print(f"=== stability check ===")
    print(f"pass: {n_pass}/{n_total}")
    print(f"fail: {n_total - n_pass}")
    print()
    
    # max-eigenvalue distribution
    valid = result_df[result_df['max_abs_eig'].notna()]
    print(f"max |eigenvalue| distribution:")
    print(f"  Min: {valid['max_abs_eig'].min():.4f}")
    print(f"  Max: {valid['max_abs_eig'].max():.4f}")
    print(f"  Mean: {valid['max_abs_eig'].mean():.4f}")
    print()
    
    # borderline keywords (close to 1)
    borderline = valid[valid['max_abs_eig'] > 0.95]
    if len(borderline) > 0:
        print(f"[!] keywords with max |eigenvalue| > 0.95 (borderline):")
        print(borderline[['keyword', 'lag', 'max_abs_eig']].to_string(
            index=False))
    else:
        print("[ok] all keywords have max |eigenvalue| <= 0.95")
    print()
    
    # failed keyword details
    failed = result_df[result_df['stability'] != 'pass']
    if len(failed) > 0:
        print(f"[x] failed keywords:")
        print(failed.to_string(index=False))
    else:
        print("[ok] all 26 keywords satisfy the stability condition")
    
    # save CSV
    result_df.to_csv(
        f'{OUTPUT_DIR}/var_stability_check.csv',
        index=False, encoding='utf-8-sig'
    )
    print(f"\nsaved: ./output_260418/var_stability_check.csv")

In [ ]:
# find the keyword with the largest |lambda|
result_main = pd.read_csv(f'{OUTPUT_DIR}/var_stability_check.csv')
max_kw = result_main.loc[result_main['max_abs_eig'].idxmax()]
print(f"max |lambda| keyword: {max_kw['keyword']}, "
      f"lag {max_kw['lag']}, |λ|={max_kw['max_abs_eig']}")

## 8. Table A2 — ADF input forms

In [ ]:
# build the input-form column for Table A2
def format_input_form(row):
    """Build the input-form string from the search_diff / news_diff flags."""
    s = row['search_diff']
    n = row['news_diff']
    if not s and not n:
        return 'level'
    elif s and not n:
        return 'first diff (S)'
    elif not s and n:
        return 'first diff (N)'
    else:
        return 'first diff (S, N)'

# filter to the 26 main keywords
df_main_adf = df_adf_results[
    df_adf_results['keyword'].isin(analysis_keywords_main)
].copy()
df_main_adf['input_form'] = df_main_adf.apply(format_input_form, axis=1)

table_a2_input = df_main_adf[['keyword', 'input_form']].sort_values('keyword')
print(table_a2_input.to_string(index=False))
table_a2_input.to_csv(f'{OUTPUT_DIR}/table_a2_input_forms.csv',
                     index=False, encoding='utf-8-sig')

## 9. Section 2.2 — preliminary Pearson correlation

In [ ]:
"""
Monthly Pearson correlation between search volume and news intensity for the 26 main keywords.
Presented as a preliminary analysis in Section 2.2.

Note: computed on log-transformed levels, regardless of ADF results.
Differenced keywords should be interpreted with caution (see footnote).
"""
from scipy import stats
import numpy as np
import pandas as pd


def compute_pearson_per_keyword(df_search, df_news, keywords):
    """
    For each keyword, the Pearson correlation between log search volume
    and log1p news intensity.
    
    Parameters
    ----------
    df_search : DataFrame (months x keywords, LOD-imputed search volume)
    df_news   : DataFrame (months x keywords, weighted news intensity)
    keywords  : list (26 main keywords)
    
    Returns
    -------
    DataFrame columns: keyword, pearson_r, p_value, n_months
    """
    results = []
    for kw in keywords:
        s = np.log(df_search[kw])       # no zeros due to LOD imputation
        n = np.log1p(df_news[kw])
        
        common = s.index.intersection(n.index)
        s_c = s.loc[common]
        n_c = n.loc[common]
        
        # variance check
        if s_c.std() == 0 or n_c.std() == 0:
            results.append({
                'keyword': kw, 'pearson_r': np.nan,
                'p_value': np.nan, 'n_months': len(common)
            })
            continue
        
        r, p = stats.pearsonr(s_c, n_c)
        results.append({
            'keyword': kw, 'pearson_r': round(r, 3),
            'p_value': round(p, 4), 'n_months': len(common)
        })
    return pd.DataFrame(results)


# run
df_pearson = compute_pearson_per_keyword(
    df_search_matched, df_news_matched, analysis_keywords_main
)

# save
df_pearson.to_csv(f'{OUTPUT_DIR}/pearson_main.csv',
                  index=False, encoding='utf-8-sig')

# summary statistics (for Section 2.2)
print("===== Pearson correlation summary (N=26) =====")
print(f"mean r:     {df_pearson['pearson_r'].mean():.3f}")
print(f"std:        {df_pearson['pearson_r'].std():.3f}")
print(f"min:        {df_pearson['pearson_r'].min():.3f}")
print(f"max:        {df_pearson['pearson_r'].max():.3f}")
n_sig = (df_pearson['p_value'] < 0.05).sum()
print(f"significant: {n_sig}/26 ({100*n_sig/26:.1f}%)")

print("\n===== correlation by keyword =====")
print(df_pearson.sort_values('pearson_r', ascending=False).to_string(index=False))

## 10. Figure A1 — IRF with 95% bootstrap confidence intervals

Manual residual bootstrap (Lütkepohl 2005, App. D.3, with centering; seed=42). Plot labels are English (Pet, Real Estate, Tourism, Game, Apartment); keyword data keys stay Korean.

In [ ]:
"""
Figure A1: IRF with 95% bootstrap confidence intervals, following
Lutkepohl (2005), Appendix D.3 (residual-based bootstrap with centering).

Procedure (Lutkepohl 2005, Appendix D.3):
(1) fit VAR -> coefficients A_1..A_p, intercept nu, residuals u_hat_t
(2) resample with replacement from centered residuals u_hat_t - u_bar -> u*_t
(3) rebuild y*_t = nu + A_1 y*_{t-1} + ... + A_p y*_{t-p} + u*_t
(4) re-fit the VAR on the bootstrap series
(5) compute the bootstrap IRF
(6) after N=500 replications, take the 2.5 and 97.5 percentiles
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.var_model import VAR

# display labels (data keys stay Korean; only the plot text is English)
LABEL = {'반려동물': 'Pet', '부동산': 'Real Estate', '관광': 'Tourism',
         '게임': 'Game', '아파트': 'Apartment'}


def setup_plot_style():
    """Plot style. All displayed text is English, so no Korean font is needed."""
    plt.rcParams['axes.unicode_minus'] = False


def manual_bootstrap_irf(df_var, lag, horizon=10, n_boot=500,
                         signif=0.05, seed=42, verbose=False):
    """
    Residual-based bootstrap IRF CI following Lutkepohl (2005), Appendix D.3.
    Includes residual centering (Step 2).

    Parameters
    ----------
    df_var : pd.DataFrame
        VAR DataFrame (columns ordered [News, Search]).
    lag : int
        VAR lag order.
    horizon : int, default 10
        IRF horizon.
    n_boot : int, default 500
        number of bootstrap replications.
    signif : float, default 0.05
        significance level.
    seed : int, default 42
        reproducibility seed.
    verbose : bool
        whether to print diagnostics.

    Returns
    -------
    tuple (ndarray, ndarray, ndarray, int)
        point: (horizon+1, K, K) point estimates
        ci_lower: (horizon+1, K, K) lower 95% CI (2.5 percentile)
        ci_upper: (horizon+1, K, K) upper 95% CI (97.5 percentile)
        n_success: number of successful replications

    Raises
    ------
    ValueError
        if fewer than 50 replications succeed.
    """
    # Step 1: fit the original model
    model = VAR(df_var)
    res = model.fit(lag)

    point = res.irf(horizon).orth_irfs  # (horizon+1, K, K)

    # inputs for the bootstrap
    y = df_var.values
    T, K = y.shape
    p = lag
    T_eff = T - p

    residuals = res.resid.values  # (T_eff, K)
    intercept = res.intercept  # (K,)
    coefs = res.coefs  # (p, K, K)

    # Step 2 (Lutkepohl D.3): centered residuals; u*_t drawn from these
    residuals_centered = residuals - residuals.mean(axis=0)

    boot_irfs = []
    n_failed = 0
    rng = np.random.RandomState(seed)

    for b in range(n_boot):
        # Step 2 (cont'd): resample with replacement from centered residuals
        resid_idx = rng.randint(0, T_eff, size=T_eff)
        resid_boot = residuals_centered[resid_idx]

        # Step 3: rebuild the bootstrap series
        y_boot = np.zeros_like(y)
        y_boot[:p] = y[:p]

        for t in range(p, T):
            pred = intercept.copy()
            for j in range(p):
                pred = pred + coefs[j] @ y_boot[t - j - 1]
            y_boot[t] = pred + resid_boot[t - p]

        # Step 4: re-fit the VAR on the bootstrap series
        try:
            df_boot = pd.DataFrame(y_boot, columns=df_var.columns,
                                    index=df_var.index)
            model_boot = VAR(df_boot)
            res_boot = model_boot.fit(p)

            # Step 5: bootstrap IRF
            irf_boot = res_boot.irf(horizon).orth_irfs
            boot_irfs.append(irf_boot)
        except Exception:
            n_failed += 1
            continue

    n_success = len(boot_irfs)
    if n_success < 50:
        raise ValueError(
            f"only {n_success}/{n_boot} bootstrap replications succeeded (< 50)."
        )

    if verbose:
        print(f"  bootstrap succeeded: {n_success}/{n_boot} ({n_failed} failed)")

    # Step 6 & CI: standard percentile interval
    boot_irfs = np.array(boot_irfs)
    alpha_low = (signif / 2) * 100       # 2.5
    alpha_high = (1 - signif / 2) * 100  # 97.5

    ci_lower = np.percentile(boot_irfs, alpha_low, axis=0)
    ci_upper = np.percentile(boot_irfs, alpha_high, axis=0)

    return point, ci_lower, ci_upper, n_success


def plot_irf_panel(ax, periods, point, lower, upper, title,
                   line_color='#1f4e79', fill_color='#7fb3e0'):
    """Draw a single IRF panel."""
    ax.fill_between(periods, lower, upper, color=fill_color,
                    alpha=0.4, zorder=1)
    ax.plot(periods, point, color=line_color, linewidth=2.0, zorder=3)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.6, zorder=2)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Period (months)', fontsize=9)
    ax.set_ylabel('Orthogonalized IRF', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.3)


def generate_figure_a1(keywords, df_search, df_news, adf_results,
                       lag_map, horizon=10, n_boot=500, seed=42,
                       output_path='output/figure_a1_irf', verbose=True):
    """Generate Figure A1 (IRF panels with 95% bootstrap CIs)."""
    setup_plot_style()

    n_kw = len(keywords)
    fig, axes = plt.subplots(n_kw, 2, figsize=(11, 2.3 * n_kw))
    if n_kw == 1:
        axes = axes.reshape(1, -1)

    for row, kw in enumerate(keywords):
        if kw not in lag_map:
            raise ValueError(f"'{kw}' not in lag_map")

        label = LABEL.get(kw, kw)
        if verbose:
            print(f"\n[{label}] computing bootstrap...")

        df_var, _ = vg.prepare_keyword_data(kw, df_search, df_news, adf_results)

        point, ci_lower, ci_upper, n_success = manual_bootstrap_irf(
            df_var, lag=lag_map[kw], horizon=horizon,
            n_boot=n_boot, seed=seed, verbose=verbose
        )

        periods = np.arange(horizon + 1)

        # N -> S
        point_ns = point[:, 1, 0]
        lower_ns = ci_lower[:, 1, 0]
        upper_ns = ci_upper[:, 1, 0]

        if verbose:
            ok = np.sum((lower_ns <= point_ns) & (point_ns <= upper_ns))
            ci_width = (upper_ns - lower_ns).mean()
            print(f"  N->S: point est. within CI {ok}/{horizon+1}, "
                  f"mean CI width {ci_width:.4f}")

        plot_irf_panel(
            axes[row, 0], periods, point_ns, lower_ns, upper_ns,
            title=f'{label}: News \u2192 Search (N \u2192 S)',
            line_color='#1f4e79', fill_color='#7fb3e0'
        )

        # S -> N
        point_sn = point[:, 0, 1]
        lower_sn = ci_lower[:, 0, 1]
        upper_sn = ci_upper[:, 0, 1]

        if verbose:
            ok = np.sum((lower_sn <= point_sn) & (point_sn <= upper_sn))
            ci_width = (upper_sn - lower_sn).mean()
            print(f"  S->N: point est. within CI {ok}/{horizon+1}, "
                  f"mean CI width {ci_width:.4f}")

        plot_irf_panel(
            axes[row, 1], periods, point_sn, lower_sn, upper_sn,
            title=f'{label}: Search \u2192 News (S \u2192 N)',
            line_color='#8b0000', fill_color='#e8a5a5'
        )

    plt.tight_layout()
    plt.savefig(f'{output_path}.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_path}.png', dpi=300, bbox_inches='tight')
    plt.close()

    print(f"\nsaved: {output_path}.pdf, {output_path}.png")


# ===== run =====
generate_figure_a1(
    keywords=['반려동물', '부동산', '관광', '게임', '아파트'],
    df_search=df_search_matched,
    df_news=df_news_matched,
    adf_results=df_adf_results,
    lag_map={'반려동물': 2, '부동산': 2, '관광': 2,
             '게임': 1, '아파트': 1},
    horizon=10,
    n_boot=500,
    seed=42,
    output_path=f'{OUTPUT_DIR}/figure_a1_irf',
    verbose=True,
)


## Appendix — methodological justification (not paper results)

`irf_resim` diagnostic motivating the manual bootstrap (statsmodels returned identical replications).

In [ ]:
"""
irf_resim diagnostic: check whether the 500 bootstrap replications actually
produce different results (motivates the manual bootstrap in bootstrap_irf_ci).
"""
import numpy as np
from statsmodels.tsa.vector_ar.var_model import VAR

kw = '반려동물'
df_var, _ = vg.prepare_keyword_data(kw, df_search_matched, df_news_matched,
                                    df_adf_results)
model = VAR(df_var)
res = model.fit(2)

# call irf_resim directly
ma_coll = res.irf_resim(orth=True, repl=500, steps=10, seed=42)
print(f"ma_coll shape: {ma_coll.shape}")  # expected: (500, 11, 2, 2)
print(f"\nStep 2, News->Search IRF distribution (500 samples):")
samples = ma_coll[:, 2, 1, 0]  # [replication, period, response, shock]
print(f"  min: {samples.min():.4f}")
print(f"  max: {samples.max():.4f}")
print(f"  mean: {samples.mean():.4f}")
print(f"  std: {samples.std():.4f}")
print(f"  2.5 percentile: {np.percentile(samples, 2.5):.4f}")
print(f"  97.5 percentile: {np.percentile(samples, 97.5):.4f}")
print(f"\nfirst 10 samples: {samples[:10]}")


### End

Complete pipeline. All outputs are written to `OUTPUT_DIR`.